# 🧠 GENESIS-Q: Gemma 4 Fine-Tuning on NVIDIA Blackwell (96GB VRAM)
### Google Colab Pro+ G4 ✕ In-Situ Antigravity Autonomous Orchestration

**ターゲット環境**:
- **GPU**: NVIDIA RTX PRO 6000 Blackwell Server Edition (97,887 MiB VRAM)
- **AI Core**: Google Gemma 4 (26B A4B MoE / 31B Dense)
- **Orchestrator**: Google Antigravity CLI (`agy`)
- **Data Source**: Transport Shield 2.0 (判例調書 0:100) & GENESIS-Q (量子コンパイラ)

In [ ]:
# 1. GPU & Google Antigravity CLI の稼働状態確認
!nvidia-smi
!agy --version

## 📁 2. Google Drive（GENESIS_ROOT）のマウント

In [ ]:
import os
from pathlib import Path
from google.colab import drive

# Google Drive をマウント
drive.mount('/content/drive')

# GENESIS_ROOT パスの設定
GENESIS_ROOT = Path('/content/drive/MyDrive/GENESIS_ROOT')
print(f"GENESIS_ROOT exists: {GENESIS_ROOT.exists()}")
DATASET_PATH = GENESIS_ROOT / 'data' / 'finetuning' / 'genesis_gemma4_instruction_tuning.jsonl'
print(f"Training dataset exists: {DATASET_PATH.exists()}")

## ⚡ 3. 高速ファインチューニング環境のセットアップ (TRL + PEFT + BitsAndBytes)

In [ ]:
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes
print("✅ Fine-Tuning libraries successfully installed.")

## 🚀 4. Gemma 4 モデルのロード (96GB VRAM 最適化)
Blackwellの96GB超大容量メモリにより、bf16高精度またはQLoRAで超高速ロードが可能です。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/gemma-2-27b-it"  # または gemma-4 公式チェックポイント

# 96GB VRAM を活用した 4bit/8bit QLoRA または bf16 ロード設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print(f"Loading {MODEL_ID} on NVIDIA Blackwell...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model = prepare_model_for_kbit_training(model)

# LoRA アダプタ設定 (GENESIS特化型)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("✅ Model and LoRA adapters successfully prepared on Blackwell GPU!")

## 📊 5. GENESIS 特化データセットの読み込み & フォーマット

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files=str(DATASET_PATH), split='train')

def format_prompts(batch):
    texts = []
    for inst, inp, out in zip(batch['instruction'], batch['input'], batch['output']):
        text = f"<start_of_turn>user\n{inst}\n\n【入力状況】\n{inp}<end_of_turn>\n<start_of_turn>model\n{out}<end_of_turn>"
        texts.append(text)
    return {'text': texts}

formatted_dataset = dataset.map(format_prompts, batched=True)
print(f"Loaded {len(formatted_dataset)} training examples.")
print("\n--- Sample Formatted Prompt ---")
print(formatted_dataset[0]['text'][:300] + "...")

## 🔥 6. SFT 学習実行 (Supervised Fine-Tuning)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = GENESIS_ROOT / "outputs" / "gemma4_genesis_q_lora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=30,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,  # Blackwell Tensor Core 最適化
    logging_steps=1,
    optim="paged_adamw_8bit",
    save_strategy="no",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

print("🚀 Starting Training on NVIDIA Blackwell (96GB VRAM)... Hold tight!")
trainer.train()

# 学習済み LoRA アダプタを Google Drive へ恒久保存
trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"\n🎉 Training Complete! LoRA weights saved directly to Google Drive:\n{OUTPUT_DIR}")

## 🧪 7. 推論テスト（GENESIS特化型知能の動作確認）

In [ ]:
prompt = """<start_of_turn>user
自転車とタクシーの右側追突事故において、直前の歩行者合図と急停車による過失割合を判例タイムズおよび因果関係に基づいて算定せよ。

【入力状況】
自転車が直進中、前方を走行するタクシーが歩道側の客待ち挙手合図に反応し、合図（ウインカー）なしに左側に急幅寄せ停車。自転車は急制動を行うも回避不能で追突した。<end_of_turn>
<start_of_turn>model
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=400, temperature=0.2)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=== Fine-Tuned Model Inference Output ===")
print(response)